## 00 — The Viewer

This is the assembly notebook. Every component we built across Modules 00–05 is brought together here into one clean, readable file.

No new concepts. The goal is:
- See all the pieces side by side
- Produce a working viewer with styled output
- Understand the full data flow in one place

Read through the code before running it. Every line should be recognizable.

## 1. Imports and Paths

In [ ]:
import json
import math
import time
from pathlib import Path
from ipyleaflet import Map, GeoJSON
import ipywidgets as widgets

LOD_DIR = Path("../../data/lod")

LOD_CONFIG = [
    {"name": "coarse",     "file": "railroads_coarse.geojson",     "zoom_max": 3},
    {"name": "medium",     "file": "railroads_medium.geojson",     "zoom_max": 6},
    {"name": "fine",       "file": "railroads_fine.geojson",       "zoom_max": 10},
    {"name": "extra_fine", "file": "railroads_extra_fine.geojson", "zoom_max": 99},
]

## 2. Geometry and Index Utilities

In [ ]:
def feature_bbox(feature):
    coords = feature["geometry"]["coordinates"]
    lons = [c[0] for c in coords]
    lats = [c[1] for c in coords]
    return [min(lons), min(lats), max(lons), max(lats)]


def leaflet_bounds_to_bbox(bounds):
    """Convert ipyleaflet [[lat_min,lon_min],[lat_max,lon_max]] to [lon_min,lat_min,lon_max,lat_max]."""
    (lat_min, lon_min), (lat_max, lon_max) = bounds
    return [lon_min, lat_min, lon_max, lat_max]


class GridIndex:
    """Uniform grid spatial index. Assigns features to 10° cells for fast viewport queries."""

    CELL_SIZE = 10.0

    def __init__(self):
        self.cells = {}

    def _cells(self, bbox):
        lon_min, lat_min, lon_max, lat_max = bbox
        cs = self.CELL_SIZE
        col_min, col_max = int((lon_min + 180) / cs), int((lon_max + 180) / cs)
        row_min, row_max = int((lat_min +  90) / cs), int((lat_max +  90) / cs)
        return [(c, r) for c in range(col_min, col_max + 1) for r in range(row_min, row_max + 1)]

    def build(self, features):
        self.cells = {}
        for idx, f in enumerate(features):
            for cell in self._cells(feature_bbox(f)):
                self.cells.setdefault(cell, []).append((idx, f))

    def query(self, viewport_bbox):
        seen, results = set(), []
        for cell in self._cells(viewport_bbox):
            for idx, f in self.cells.get(cell, []):
                if idx not in seen:
                    seen.add(idx)
                    results.append(f)
        return results

## 3. Load Data and Build Indexes

In [ ]:
lod_indexes = {}

for cfg in LOD_CONFIG:
    t0 = time.perf_counter()
    with open(LOD_DIR / cfg["file"]) as f:
        features = json.load(f)["features"]
    idx = GridIndex()
    idx.build(features)
    elapsed = time.perf_counter() - t0
    lod_indexes[cfg["name"]] = idx
    print(f"  {cfg['name']:<12}  {len(features):>6,} features  {elapsed:.2f}s")

print("\nReady.")

## 4. LOD Selection

In [ ]:
def get_lod(zoom):
    z = int(math.floor(zoom))
    for cfg in LOD_CONFIG:
        if z <= cfg["zoom_max"]:
            return cfg["name"]
    return LOD_CONFIG[-1]["name"]

## 5. Styling

We use `scalerank` to vary line weight — more important lines render thicker, just like a real map.

In [ ]:
def style_callback(feature):
    rank = feature["properties"].get("scalerank", 5)
    weight = max(0.5, 2.5 - rank * 0.3)   # rank 1 → thick, rank 7 → thin
    return {
        "color":   "#333333",
        "weight":  weight,
        "opacity": 0.85,
    }

## 6. The Map

In [ ]:
current_lod = get_lod(5)

m = Map(center=[48.5, 10.0], zoom=5)

layer = GeoJSON(
    data={"type": "FeatureCollection", "features": []},
    style_callback=style_callback
)
m.add(layer)

# Status bar
lbl_lod      = widgets.Label()
lbl_zoom     = widgets.Label()
lbl_features = widgets.Label()
lbl_time     = widgets.Label()
status       = widgets.HBox([lbl_lod, lbl_zoom, lbl_features, lbl_time])

def update(*args):
    global current_lod
    if not m.bounds:
        return

    current_lod    = get_lod(m.zoom)
    vp             = leaflet_bounds_to_bbox(m.bounds)

    t0             = time.perf_counter()
    visible        = lod_indexes[current_lod].query(vp)
    elapsed_ms     = (time.perf_counter() - t0) * 1000

    layer.data          = {"type": "FeatureCollection", "features": visible}
    lbl_lod.value       = f"LOD: {current_lod}"
    lbl_zoom.value      = f"  zoom: {int(math.floor(m.zoom))}"
    lbl_features.value  = f"  features: {len(visible):,}"
    lbl_time.value      = f"  query: {elapsed_ms:.1f}ms"

m.observe(update, names=["zoom", "bounds"])
update()

widgets.VBox([m, status])

## Exercise A

Add a second style dimension: color the lines by `category` property.

1. Find the unique `category` values in the fine LOD file
2. Assign a distinct color to each
3. Update `style_callback` to use category color + scalerank weight together

The result should show the railroad network colored by line type.

In [ ]:
CATEGORY_COLORS = {
    0: '#264653',
    1: '#2a9d8f',
    2: '#e76f51',
    3: '#e9c46a',
    6: '#577590',
    7: '#8ab17d',
    9: '#bc4749',
}

def style_callback(feature):
    props = feature['properties']
    rank = props.get('scalerank', 5)
    category = props.get('category')
    weight = max(0.6, 2.8 - rank * 0.25)
    return {
        'color': CATEGORY_COLORS.get(category, '#333333'),
        'weight': weight,
        'opacity': 0.9,
    }

layer.style_callback = style_callback
update()
widgets.VBox([m, status])


## Exercise B

Add a tooltip: when the user hovers over a railroad feature, show its `category` and `scalerank` in a widget below the map.

Hint: use the GeoJSON layer's `on_hover` event.

In [ ]:
info = widgets.HTML(value='Hover over a railroad to inspect its properties.')

def handle_hover(event=None, feature=None, properties=None, **kwargs):
    props = properties or (feature or {}).get('properties', {})
    category = props.get('category', '—')
    scalerank = props.get('scalerank', '—')
    info.value = f'<b>category:</b> {category} &nbsp;&nbsp; <b>scalerank:</b> {scalerank}'

try:
    layer.on_hover(handle_hover)
    print('Hover callback attached.')
except AttributeError:
    info.value = 'This ipyleaflet build does not expose GeoJSON.on_hover().' 

widgets.VBox([m, status, info])


## Check Your Understanding

The viewer re-runs `get_lod()` and queries the grid on **every** bounds change — even if the user only pans a few pixels and the LOD hasn't changed.

Describe two optimizations you could apply to skip redundant work:
1. One that avoids calling `get_lod()` unnecessarily
2. One that avoids re-querying the grid when the viewport barely moved

For each, what is the tradeoff?

One optimization is to cache the last integer zoom or last chosen LOD and skip recomputing the decision until that discrete value changes; the tradeoff is slightly more state bookkeeping. Another is to cache the last viewport bbox and only re-query when the map crosses a movement threshold or cell boundary, which saves work but allows slightly stale feature counts during tiny pans.


## Next

In [01 — What We Built](./01-What_We_Built.ipynb), we step back, measure the system's remaining limitations, and document every decision we made.